### This file is handling sector column. Since the project later involves showing analysis based on sectors, this file create new feature called sector which will show sector for each property. 
### It also handles some redundant features.

In [1]:
import pandas as pd
import numpy as np
import re

In [3]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [4]:
df = pd.read_csv('gurgaon_data.csv')
df.sample(3)

,property_name,property_type,society,price,price_sq.feet,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,address,floorNum,facing,agePossession,nearbyLocations,description,furnishDetails,features,rating
1090,2 BHK Flat in Sector 95 Gurgaon,flat,sidhartha ncr one,0.56,4106.0,1364.0,Built Up area: 1364 (126.72 sq.m.)Carpet area:...,2,2,2,not available,"Sector 95 Gurgaon, Gurgaon, Haryana",10.0,NaN,undefined,NaN,"2 bath, unfurnished, 10th floor (Of 17), at se...",NaN,NaN,"['Environment4 out of 5', 'Lifestyle4.5 out of..."
1294,3 BHK Flat in Sector 69 Gurgaon,flat,tulip violet,1.32,8411.0,1569.0,Super Built up area 1568(145.67 sq.m.),3,3,1,pooja room,"12th Floor, Sector 69 Gurgaon, Gurgaon, Haryana",12.0,South-West,1 to 5 Year Old,"['Airia Mall Sector 68', 'Southern Peripheral ...",Tulip violet is one of gurgaon's most sought a...,"['2 Wardrobe', '6 Fan', '1 Exhaust Fan', '9 Li...","['Power Back-up', 'Feng Shui / Vaastu Complian...","['Green Area5 out of 5', 'Construction4 out of..."
497,3 BHK Flat in Sector 37D Gurgaon,flat,ramprastha primera,1.02,6000.0,1700.0,Super Built up area 1700(157.94 sq.m.),3,2,3,not available,"Sector 37D Gurgaon, Gurgaon, Haryana",22.0,NaN,Under Construction,"['JMS Marine Square Mall', 'HUDA Market', 'Dwa...",Residential apartment for sell.The property ha...,[],"['Power Back-up', 'Feng Shui / Vaastu Complian...","['Safety4 out of 5', 'Lifestyle4 out of 5', 'E..."


In [5]:
df.shape

(3941, 20)

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3941 entries, 0 to 3940
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   property_name    3941 non-null   object 
 1   property_type    3941 non-null   object 
 2   society          3940 non-null   object 
 3   price            3941 non-null   float64
 4   price_sq.feet    3941 non-null   float64
 5   area             3941 non-null   float64
 6   areaWithType     3941 non-null   object 
 7   bedRoom          3941 non-null   int64  
 8   bathroom         3941 non-null   int64  
 9   balcony          3941 non-null   object 
 10  additionalRoom   3941 non-null   object 
 11  address          3930 non-null   object 
 12  floorNum         3922 non-null   float64
 13  facing           2770 non-null   object 
 14  agePossession    3940 non-null   object 
 15  nearbyLocations  3735 non-null   object 
 16  description      3941 non-null   object 
 17  furnishDetails

### Lets check null values

In [9]:
df.isnull().sum()

property_name         0
property_type         0
society               1
price                 0
price_sq.feet         0
area                  0
areaWithType          0
bedRoom               0
bathroom              0
balcony               0
additionalRoom        0
address              11
floorNum             19
facing             1171
agePossession         1
nearbyLocations     206
description           0
furnishDetails     1026
features            705
rating              448
dtype: int64

### Getting sector info from property_name feature

In [10]:
df.head(3)

,property_name,property_type,society,price,price_sq.feet,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,address,floorNum,facing,agePossession,nearbyLocations,description,furnishDetails,features,rating
0,3 BHK Flat in Sector 70A Gurgaon,flat,bptp astaire gardens,1.20,7058.0,1700.0,Super Built up area 1700(157.94 sq.m.)Built Up...,3,3,3+,not available,"Sector 70A Gurgaon, Gurgaon, Haryana",2.0,NaN,1 to 5 Year Old,"['Sector 54 Chowk Metro Station', 'Airia Mall'...",Having very good quality of wooden work and it...,"['5 Fan', '1 Exhaust Fan', '1 Dining Table', '...","['Intercom Facility', 'Lift(s)', 'Swimming Poo...","['Green Area4 out of 5', 'Construction5 out of..."
1,8 Bedroom House for sale in Sector 56 Gurgaon,house,independent,7.50,23148.0,3240.0,Plot area 360(301.01 sq.m.),8,8,3+,"pooja room,study room,servant room","Sector 56 Gurgaon, Gurgaon, Haryana",3.0,North-East,1 to 5 Year Old,"['Sector metro station', 'Sector metro station...",8 bhk luxury kothi for sale in sector -56 clos...,"['1 Water Purifier', '25 Fan', '1 Fridge', '1 ...","['Feng Shui / Vaastu Compliant', 'Private Gard...","['Environment4 out of 5', 'Safety4 out of 5', ..."
2,3 BHK Flat in Sector 83 Gurgaon,flat,vatika lifestyle homes,1.15,8214.0,1400.0,Super Built up area 1832(170.2 sq.m.)Carpet ar...,3,3,1,not available,"Block A, Sector 83 Gurgaon, Gurgaon, Haryana",7.0,South-West,5 to 10 Year Old,"['Sapphire 83 Mall', 'NH – 08', 'Knowledge Tre...",For the actual picture please let us know if t...,"['3 Wardrobe', '2 Fan', '1 Exhaust Fan', '2 Ge...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...","['Green Area4.5 out of 5', 'Construction4 out ..."


In [11]:
# Creating new feature
df.insert(loc = 3, column= 'sector', value= df['property_name'].str.split('in').str.get(1).str.replace('Gurgaon', '').str.strip())

In [12]:
df.head()

,property_name,property_type,society,sector,price,price_sq.feet,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,address,floorNum,facing,agePossession,nearbyLocations,description,furnishDetails,features,rating
0,3 BHK Flat in Sector 70A Gurgaon,flat,bptp astaire gardens,Sector 70A,1.20,7058.0,1700.0,Super Built up area 1700(157.94 sq.m.)Built Up...,3,3,3+,not available,"Sector 70A Gurgaon, Gurgaon, Haryana",2.0,NaN,1 to 5 Year Old,"['Sector 54 Chowk Metro Station', 'Airia Mall'...",Having very good quality of wooden work and it...,"['5 Fan', '1 Exhaust Fan', '1 Dining Table', '...","['Intercom Facility', 'Lift(s)', 'Swimming Poo...","['Green Area4 out of 5', 'Construction5 out of..."
1,8 Bedroom House for sale in Sector 56 Gurgaon,house,independent,Sector 56,7.50,23148.0,3240.0,Plot area 360(301.01 sq.m.),8,8,3+,"pooja room,study room,servant room","Sector 56 Gurgaon, Gurgaon, Haryana",3.0,North-East,1 to 5 Year Old,"['Sector metro station', 'Sector metro station...",8 bhk luxury kothi for sale in sector -56 clos...,"['1 Water Purifier', '25 Fan', '1 Fridge', '1 ...","['Feng Shui / Vaastu Compliant', 'Private Gard...","['Environment4 out of 5', 'Safety4 out of 5', ..."
2,3 BHK Flat in Sector 83 Gurgaon,flat,vatika lifestyle homes,Sector 83,1.15,8214.0,1400.0,Super Built up area 1832(170.2 sq.m.)Carpet ar...,3,3,1,not available,"Block A, Sector 83 Gurgaon, Gurgaon, Haryana",7.0,South-West,5 to 10 Year Old,"['Sapphire 83 Mall', 'NH – 08', 'Knowledge Tre...",For the actual picture please let us know if t...,"['3 Wardrobe', '2 Fan', '1 Exhaust Fan', '2 Ge...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...","['Green Area4.5 out of 5', 'Construction4 out ..."
3,6 Bedroom House for sale in Laxmi Garden,house,independent,Laxmi Garden,2.10,17284.0,1215.0,Plot area 135(112.88 sq.m.),6,6,3+,not available,"Near Tiokna Park, Laxmi Garden, Gurgaon, Haryana",3.0,NaN,1 to 5 Year Old,"['Rajiv Chowk Mosque', 'Rachna Dental Clinic',...",Triple story corner big house size 30 feet x 4...,"['3 Chimney', 'No AC', 'No Bed', 'No Curtains'...","['Water Storage', 'Visitor Parking', 'Waste Di...","['Environment4 out of 5', 'Lifestyle4 out of 5..."
4,3 BHK Flat in Sector 102 Gurgaon,flat,emaar gurgaon greens,Sector 102,1.40,8484.0,1650.0,Super Built up area 1650(153.29 sq.m.)Built Up...,3,3,3,servant room,"Sector 102 Gurgaon, Gurgaon, Haryana",10.0,East,1 to 5 Year Old,"['JMS Marine Square Mall', 'Dwarka Expressway'...",Beautiful 3 bhk servant available for sale in ...,"['3 AC', 'No Bed', 'No Chimney', 'No Curtains'...","['Power Back-up', 'Intercom Facility', 'Lift(s...","['Green Area5 out of 5', 'Construction4 out of..."


In [13]:
df['sector'] = df['sector'].str.lower()

In [14]:
df['sector'].value_counts()

sector
sohna                                                        163
sector 102                                                   113
sector 85                                                    110
sector 92                                                    104
sector 69                                                     94
sector 81                                                     90
sector 65                                                     90
sector 90                                                     90
sector 109                                                    88
sector 79                                                     80
sector 83                                                     69
sector 37d                                                    68
sector 86                                                     67
sector 104                                                    66
sector 107                                                    60
sector 108        

#### This is the tricky part. Since many just have sector name instead of no., i need to manually fix this.

In [15]:
df['sector'] = df['sector'].str.replace('dharam colony','sector 12')
df['sector'] = df['sector'].str.replace('krishna colony','sector 7')
df['sector'] = df['sector'].str.replace('suncity','sector 54')
df['sector'] = df['sector'].str.replace('prem nagar','sector 13')
df['sector'] = df['sector'].str.replace('mg road','sector 28')
df['sector'] = df['sector'].str.replace('gandhi nagar','sector 28')
df['sector'] = df['sector'].str.replace('laxmi garden','sector 11')
df['sector'] = df['sector'].str.replace('shakti nagar','sector 11')
df['sector'] = df['sector'].str.replace('baldev nagar','sector 7')
df['sector'] = df['sector'].str.replace('shivpuri','sector 7')
df['sector'] = df['sector'].str.replace('garhi harsaru','sector 17')
df['sector'] = df['sector'].str.replace('imt manesar','manesar')
df['sector'] = df['sector'].str.replace('adarsh nagar','sector 12')
df['sector'] = df['sector'].str.replace('shivaji nagar','sector 11')
df['sector'] = df['sector'].str.replace('bhim nagar','sector 6')
df['sector'] = df['sector'].str.replace('madanpuri','sector 7')
df['sector'] = df['sector'].str.replace('saraswati vihar','sector 28')
df['sector'] = df['sector'].str.replace('arjun nagar','sector 8')
df['sector'] = df['sector'].str.replace('ravi nagar','sector 9')
df['sector'] = df['sector'].str.replace('vishnu garden','sector 105')
df['sector'] = df['sector'].str.replace('bhondsi','sector 11')
df['sector'] = df['sector'].str.replace('surya vihar','sector 21')
df['sector'] = df['sector'].str.replace('devilal colony','sector 9')
df['sector'] = df['sector'].str.replace('valley view estate','gwal pahari')

In [16]:
df['sector'] = df['sector'].str.replace('mehrauli  road','sector 14')
df['sector'] = df['sector'].str.replace('jyoti park','sector 7')
df['sector'] = df['sector'].str.replace('ansal plaza','sector 23')
df['sector'] = df['sector'].str.replace('dayanand colony','sector 6')
df['sector'] = df['sector'].str.replace('sushant lok phase 2','sector 55')
df['sector'] = df['sector'].str.replace('chakkarpur','sector 28')
df['sector'] = df['sector'].str.replace('greenwood city','sector 45')
df['sector'] = df['sector'].str.replace('subhash nagar','sector 12')
df['sector'] = df['sector'].str.replace('sushant lok phase 3','sector 57')
df['sector'] = df['sector'].str.replace('dlf phase 5','sector 43')
df['sector'] = df['sector'].str.replace('rajendra park','sector 105')
df['sector'] = df['sector'].str.replace('uppals southend','sector 49')
df['sector'] = df['sector'].str.replace('sohna','sohna road')
df['sector'] = df['sector'].str.replace('ashok vihar phase 3 extension','sector 5')
df['sector'] = df['sector'].str.replace('south city 1','sector 41')
df['sector'] = df['sector'].str.replace('ashok vihar phase 2','sector 5')
df['sector'] = df['sector'].str.replace('sohna road road','sohna road')
df['sector'] = df['sector'].str.replace('malibu town','sector 47')
df['sector'] = df['sector'].str.replace('surat nagar 1','sector 104')
df['sector'] = df['sector'].str.replace('new colony','sector 7')
df['sector'] = df['sector'].str.replace('mianwali colony','sector 12')
df['sector'] = df['sector'].str.replace('jacobpura','sector 12')
df['sector'] = df['sector'].str.replace('rajiv nagar','sector 13')
df['sector'] = df['sector'].str.replace('ashok vihar','sector 3')
df['sector'] = df['sector'].str.replace('dlf phase 1','sector 26')
df['sector'] = df['sector'].str.replace('nirvana country','sector 50')
df['sector'] = df['sector'].str.replace('palam vihar','sector 2')
df['sector'] = df['sector'].str.replace('dlf phase 2','sector 25')
df['sector'] = df['sector'].str.replace('sushant lok phase 1','sector 43')
df['sector'] = df['sector'].str.replace('laxman vihar','sector 4')
df['sector'] = df['sector'].str.replace('dlf phase 4','sector 28')
df['sector'] = df['sector'].str.replace('dlf phase 3','sector 24')

In [17]:
df['sector'].value_counts()

sector
sohna road                                                   175
sector 102                                                   113
sector 85                                                    110
sector 92                                                    104
sector 69                                                     94
sector 90                                                     90
sector 81                                                     90
sector 65                                                     90
sector 109                                                    88
sector 79                                                     80
sector 104                                                    73
sector 83                                                     69
sector 37d                                                    68
sector 86                                                     67
sector 50                                                     64
sector 107        

In [18]:
df['sector'] = df['sector'].str.replace('sector 95a','sector 95')
df['sector'] = df['sector'].str.replace('sector 23a','sector 23')
df['sector'] = df['sector'].str.replace('sector 12a','sector 12')
df['sector'] = df['sector'].str.replace('sector 3a','sector 3')
df['sector'] = df['sector'].str.replace('sector 110 a','sector 110')
df['sector'] = df['sector'].str.replace('patel nagar','sector 15')
df['sector'] = df['sector'].str.replace('a block sector 43','sector 43')
df['sector'] = df['sector'].str.replace('maruti kunj','sector 12')
df['sector'] = df['sector'].str.replace('b block sector 43','sector 43')
df['sector'] = df['sector'].str.replace('sector-33 sohna road','sector 33')
df['sector'] = df['sector'].str.replace('sector 1 manesar','manesar')
df['sector'] = df['sector'].str.replace('sector 4 phase 2','sector 4')
df['sector'] = df['sector'].str.replace('sector 1a manesar','manesar')
df['sector'] = df['sector'].str.replace('c block sector 43','sector 43')
df['sector'] = df['sector'].str.replace('sector 89 a','sector 89')
df['sector'] = df['sector'].str.replace('sector 2 extension','sector 2')
df['sector'] = df['sector'].str.replace('sector 36 sohna road','sector 36')

In [19]:
df['sector'].value_counts()

sector
sohna road                                                   175
sector 102                                                   113
sector 85                                                    110
sector 92                                                    104
sector 69                                                     94
sector 90                                                     90
sector 81                                                     90
sector 65                                                     90
sector 109                                                    88
sector 79                                                     80
sector 104                                                    73
sector 33                                                     71
sector 83                                                     69
sector 37d                                                    68
sector 86                                                     67
sector 43         

In [20]:
df[df['sector'] == 'new sector 2']

,property_name,property_type,society,sector,price,price_sq.feet,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,address,floorNum,facing,agePossession,nearbyLocations,description,furnishDetails,features,rating
1232,2 BHK Flat in New Palam Vihar,flat,my home,new sector 2,0.28,3166.0,884.0,Carpet area: 900 (83.61 sq.m.),2,1,1,others,"F 150/b, New Palam Vihar, Gurgaon, Haryana",2.0,NaN,1 to 5 Year Old,"['Palam Vihar Vyapar kendra', 'Palam triangle'...","2 bhk room with wooden coverd ,1 drawing room,...","['3 Wardrobe', '5 Light', '1 Chimney', '1 Modu...","['Water Storage', 'Park']","['Environment4 out of 5', 'Safety4 out of 5', ..."
1434,2 Bedroom House for sale in New Palam Vihar,house,my home,new sector 2,0.34,12592.0,270.0,Plot area 270(25.08 sq.m.),2,2,2,not available,"Ez-19 A, New Palam Vihar, Gurgaon, Haryana",3.0,West,5 to 10 Year Old,"['Palam Vihar Vyapar kendra', 'Palam triangle'...",There are availability of various facilities l...,"['1 Wardrobe', '3 Fan', '6 Light', 'No AC', 'N...","['Water Storage', 'Park', 'Visitor Parking']","['Environment4 out of 5', 'Lifestyle4 out of 5..."
2164,2 BHK Flat in New Palam Vihar,flat,ompee k s residency,new sector 2,1.60,26936.0,594.0,Carpet area: 66 (55.18 sq.m.),2,2,2,not available,"New Palam Vihar, Gurgaon, Haryana",1.0,NaN,1 to 5 Year Old,"['Palam Vihar Vyapar kendra', 'Palam triangle'...",We are the proud owners of this 2 bhk apartmen...,NaN,NaN,"['Environment4 out of 5', 'Safety4 out of 5', ..."
3418,2 BHK Flat in New Palam Vihar,flat,my home,new sector 2,0.22,4400.0,500.0,Carpet area: 500 (46.45 sq.m.),2,2,1,not available,"New Palam Vihar, Gurgaon, Haryana",1.0,NaN,0 to 1 Year Old,"['Palam Vihar Vyapar kendra', 'Palam triangle'...",Cctv surveillance are provided here. There is ...,"['3 Fan', '1 Exhaust Fan', '15 Light', '1 Modu...",NaN,"['Safety4 out of 5', 'Lifestyle4 out of 5', 'E..."
3858,3 Bedroom House for sale in New Palam Vihar,house,independent,new sector 2,1.00,8796.0,1137.0,Plot area 120(100.34 sq.m.)Built Up area: 120 ...,3,2,2,pooja room,"Q-148, New Palam Vihar, Phase-2, Near Royal Oa...",1.0,North,10+ Year Old,"['Palam Vihar Vyapar kendra', 'Palam triangle'...","Ground and first floor, Ground floor: Ground f...",NaN,NaN,"['Environment4 out of 5', 'Lifestyle4 out of 5..."


In [21]:
df.loc[[1232, 1434, 2164, 3418, 3858], 'sector'] = 'sector 110'

In [22]:
df[df['sector'] == 'new']

,property_name,property_type,society,sector,price,price_sq.feet,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,address,floorNum,facing,agePossession,nearbyLocations,description,furnishDetails,features,rating
662,2 BHK Flat in New Gurgaon,flat,takshila heights sector 37 c,new,0.67,5583.0,1200.0,Super Built up area 1200(111.48 sq.m.),2,2,2,not available,"New Gurgaon, Gurgaon, Haryana",3.0,NaN,1 to 5 Year Old,"['Shri Balaji Hospital and Trauma Center', 'S....",Check out this 2 bhk apartment for sale in tak...,[],"['Lift(s)', 'Swimming Pool', 'Visitor Parking'...",NaN
2537,2 BHK Flat in New Gurgaon,flat,green court,new,0.38,5507.0,690.0,Carpet area: 690 (64.1 sq.m.),2,2,1,not available,"New Gurgaon, Gurgaon, Haryana",7.0,NaN,Under Construction,"['Ing bank ATM', 'Dcb bank ATM', 'Indus ind ba...",We are the proud owners of this 2 bhk apartmen...,[],"['Intercom Facility', 'Lift(s)', 'Maintenance ...",NaN
2931,4 BHK Flat in New Gurgaon,flat,dlf 76,new,4.00,11428.0,3500.0,Carpet area: 3500 (325.16 sq.m.),4,4,2,"study room,servant room","New Gurgaon, Gurgaon, Haryana",4.0,NaN,Jun 2027,"['Shri Balaji Hospital and Trauma Center', 'S....",This lovely 4 bhk apartment/flat in new gurgao...,"['6 Wardrobe', '1 Fridge', '8 Fan', '1 Exhaust...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...",NaN
3281,4 BHK Flat in New Gurgaon,flat,sare homes,new,0.85,4786.0,1776.0,Super Built up area 1776(165 sq.m.),4,4,3,not available,"New Gurgaon, Gurgaon, Haryana",3.0,NaN,5 to 10 Year Old,"['Columbia Asia Hospital', 'Apex Multi Special...",Located in the popular residential address of ...,[],NaN,NaN


In [23]:
df.loc[662,'sector'] = 'sector 37'
df.loc[3281,'sector'] = 'sector 92'
df.loc[2537,'sector'] = 'sector 90'
df.loc[2931,'sector'] = 'sector 76'

In [24]:
df['sector'].value_counts()

sector
sohna road                                                   175
sector 102                                                   113
sector 85                                                    110
sector 92                                                    105
sector 69                                                     94
sector 90                                                     91
sector 65                                                     90
sector 81                                                     90
sector 109                                                    88
sector 79                                                     80
sector 104                                                    73
sector 33                                                     71
sector 83                                                     69
sector 37d                                                    68
sector 86                                                     67
sector 43         

In [26]:
df['sector'].isnull().sum()

np.int64(0)

In [28]:
df.head()

,property_name,property_type,society,sector,price,price_sq.feet,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,address,floorNum,facing,agePossession,nearbyLocations,description,furnishDetails,features,rating
0,3 BHK Flat in Sector 70A Gurgaon,flat,bptp astaire gardens,sector 70a,1.20,7058.0,1700.0,Super Built up area 1700(157.94 sq.m.)Built Up...,3,3,3+,not available,"Sector 70A Gurgaon, Gurgaon, Haryana",2.0,NaN,1 to 5 Year Old,"['Sector 54 Chowk Metro Station', 'Airia Mall'...",Having very good quality of wooden work and it...,"['5 Fan', '1 Exhaust Fan', '1 Dining Table', '...","['Intercom Facility', 'Lift(s)', 'Swimming Poo...","['Green Area4 out of 5', 'Construction5 out of..."
1,8 Bedroom House for sale in Sector 56 Gurgaon,house,independent,sector 56,7.50,23148.0,3240.0,Plot area 360(301.01 sq.m.),8,8,3+,"pooja room,study room,servant room","Sector 56 Gurgaon, Gurgaon, Haryana",3.0,North-East,1 to 5 Year Old,"['Sector metro station', 'Sector metro station...",8 bhk luxury kothi for sale in sector -56 clos...,"['1 Water Purifier', '25 Fan', '1 Fridge', '1 ...","['Feng Shui / Vaastu Compliant', 'Private Gard...","['Environment4 out of 5', 'Safety4 out of 5', ..."
2,3 BHK Flat in Sector 83 Gurgaon,flat,vatika lifestyle homes,sector 83,1.15,8214.0,1400.0,Super Built up area 1832(170.2 sq.m.)Carpet ar...,3,3,1,not available,"Block A, Sector 83 Gurgaon, Gurgaon, Haryana",7.0,South-West,5 to 10 Year Old,"['Sapphire 83 Mall', 'NH – 08', 'Knowledge Tre...",For the actual picture please let us know if t...,"['3 Wardrobe', '2 Fan', '1 Exhaust Fan', '2 Ge...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...","['Green Area4.5 out of 5', 'Construction4 out ..."
3,6 Bedroom House for sale in Laxmi Garden,house,independent,sector 11,2.10,17284.0,1215.0,Plot area 135(112.88 sq.m.),6,6,3+,not available,"Near Tiokna Park, Laxmi Garden, Gurgaon, Haryana",3.0,NaN,1 to 5 Year Old,"['Rajiv Chowk Mosque', 'Rachna Dental Clinic',...",Triple story corner big house size 30 feet x 4...,"['3 Chimney', 'No AC', 'No Bed', 'No Curtains'...","['Water Storage', 'Visitor Parking', 'Waste Di...","['Environment4 out of 5', 'Lifestyle4 out of 5..."
4,3 BHK Flat in Sector 102 Gurgaon,flat,emaar gurgaon greens,sector 102,1.40,8484.0,1650.0,Super Built up area 1650(153.29 sq.m.)Built Up...,3,3,3,servant room,"Sector 102 Gurgaon, Gurgaon, Haryana",10.0,East,1 to 5 Year Old,"['JMS Marine Square Mall', 'Dwarka Expressway'...",Beautiful 3 bhk servant available for sale in ...,"['3 AC', 'No Bed', 'No Chimney', 'No Curtains'...","['Power Back-up', 'Intercom Facility', 'Lift(s...","['Green Area5 out of 5', 'Construction4 out of..."


### features to drop -> property_name, address, description, rating

In [29]:
df.drop(columns=['property_name', 'address', 'description', 'rating'],inplace=True)

In [30]:
df.sample(5)

,property_type,society,sector,price,price_sq.feet,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,floorNum,facing,agePossession,nearbyLocations,furnishDetails,features
2129,flat,godrej,sector 79,1.33,8531.0,1559.0,Super Built up area 1559(144.84 sq.m.),2,2,3,study room,6.0,North-West,1 to 5 Year Old,"['Vatika Town Square-INXT', 'Naurangpur Road',...",NaN,"['Centrally Air Conditioned', 'Security / Fire..."
2258,house,independent,near euro,1.70,18889.0,900.0,Plot area 100(83.61 sq.m.)Built Up area: 61 sq...,5,4,3+,not available,3.0,East,5 to 10 Year Old,"['Shri Multispeciality Hospital', 'Kamla Hospi...","['1 Water Purifier', '7 Fan', '1 Geyser', '20 ...","['Feng Shui / Vaastu Compliant', 'Water Storag..."
212,flat,puri emerald bay,sector 104,2.25,10465.0,2150.0,Super Built up area 2450(227.61 sq.m.)Carpet a...,3,4,3+,servant room,4.0,North,1 to 5 Year Old,"['Sector-21 Metro Dwarka', 'Gurgaon Dreamz Mal...","['6 Fan', '1 Exhaust Fan', '16 Light', '5 AC',...","['Centrally Air Conditioned', 'Water purifier'..."
1214,flat,bptp freedom park life,sector 57,5.50,8982.0,6123.0,Built Up area: 5010 (465.44 sq.m.),5,6,3+,"pooja room,servant room",19.0,East,5 to 10 Year Old,"['Presidium School Gurgoan', 'Manav Rachna Sch...","['1 Water Purifier', '12 Fan', '1 Exhaust Fan'...","['Water purifier', 'Security / Fire Alarm', 'P..."
1841,flat,eldeco accolade,sohna road,0.73,5775.0,1264.0,Super Built up area 1264(117.43 sq.m.)Carpet a...,2,2,3,pooja room,5.0,West,1 to 5 Year Old,"['Global City Centre', 'Sohna Road', 'Damdama ...",NaN,"['Feng Shui / Vaastu Compliant', 'Intercom Fac..."


In [31]:
df.to_csv('gurgaon_data_v2.csv',index=False)